# Step 3 — Feature selection

| | |
|---|---|
| **Goal** | Keep a small set of informative, non-redundant features for the GAN and the IDS models. |
| **Inputs** | `data/processed/train.parquet` (selection), `val.parquet` (choice of the number of features) |
| **Outputs** | `data/processed/selected_features.json`, tables `results/tables/03_*.csv`, figures `results/figures/03_*.png` |
| **Code** | `src/features.py` |

### Why select features?
- **Redundancy:** many CICFlowMeter features are copies of each other (ρ = 1: `Tot Fwd Pkts` ↔ `Subflow Fwd Pkts`, …).
  They add nothing, slow training, and *split* feature importance between twins, which makes explanations misleading.
- **The GAN:** a generator has to learn the joint distribution of *every* feature. Fewer, non-redundant features make
  this task much easier, especially with only ~600 Web flows to learn from.
- **Interpretability and deployment:** a SOC analyst can reason about 15–20 features, not about 70.

### Method (and the leakage rule)
Everything is **fitted on the training split only**. The validation split is used once, to choose *how many* features
to keep. The test split is not opened at all in this step.

1. **Candidates:** the 70 non-constant features **minus `Dst Port`** (decision D5: testbed shortcut, measured in §3).
2. **Redundancy filter:** for every pair with |Spearman ρ| ≥ 0.95, drop the feature with the lower *mutual information*
   with the target.
3. **Ranking:** combine two families of methods, so the result does not depend on a single one:
   - *filter*: **mutual information** (MI), model-independent and able to capture non-linear dependence;
   - *embedded*: **Random Forest importance** (mean decrease in impurity), which captures interactions between features.
   Final rank = average of the two ranks.
4. **Choice of k:** validation macro-F1 of a Random Forest using the top-k features, for k = 5, 10, 15, …
   We keep the **smallest k within 0.005 of the best score** (parsimony).

**Stratified subsample:** at most 30,000 training flows per family (all 597 Web flows). With 1 M Benign flows, a plain sample
would drown the rare families, and selection would only optimise Benign-vs-rest.

## 1. Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/dija040/ids-project.git"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/ids-project")

    def git(*args):
        # Run a git command; if it fails, show git's own error message.
        r = subprocess.run(["git", *args], capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError("git " + " ".join(args) + " failed: " + r.stderr)
        return r.stdout.strip()

    if not (PROJECT_DIR / ".git").exists():
        git("clone", REPO_URL, str(PROJECT_DIR))
    else:
        # The Drive copy is only used to RUN the code (we edit it on the PC and push to
        # GitHub), so we force it to match GitHub exactly instead of merging with `pull`.
        # Untracked files (data/, generated results) are not touched.
        git("-C", str(PROJECT_DIR), "fetch", "origin")
        git("-C", str(PROJECT_DIR), "reset", "--hard", "origin/main")
    print("Code version:", git("-C", str(PROJECT_DIR), "log", "-1", "--format=%h %s"))
else:
    PROJECT_DIR = Path.cwd().resolve().parent
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src import config, features as fs
config.set_seed(); config.ensure_dirs()
pd.set_option("display.max_rows", 100, "display.width", 200)
FIG, TAB = config.FIGURES_DIR, config.TABLES_DIR

## 2. Stratified subsamples of train and validation

In [ ]:
cols = config.MODEL_FEATURES + [fs.TARGET]
tr = fs.stratified_subsample(fs.load_split("train", cols), fs.TARGET, 30_000)
va = fs.stratified_subsample(fs.load_split("val", cols), fs.TARGET, 20_000)
y_tr, y_va = tr[fs.TARGET], va[fs.TARGET]
CANDIDATES = [c for c in config.MODEL_FEATURES if c != "Dst Port"]
print(f"train subsample: {len(tr):,} flows | validation subsample: {len(va):,} flows")
pd.DataFrame({"train": y_tr.value_counts(), "val": y_va.value_counts()}).reindex(config.FAMILIES)

## 3. Evidence for D5: how much do models rely on `Dst Port`?

A Random Forest on all 70 features, `Dst Port` included. If `Dst Port` ranks at the top, the model is
mostly learning "which service this testbed attacked", a shortcut that an attacker defeats by changing port.

In [ ]:
imp_all = fs.rf_importance(tr[config.MODEL_FEATURES], y_tr)
rank_port = list(imp_all.index).index("Dst Port") + 1
print(f"Dst Port: rank {rank_port} / {len(imp_all)}, importance {imp_all['Dst Port']:.3f} "
      f"(a uniform share would be {1/len(imp_all):.3f})")
imp_all.head(10).to_frame()

## 4. Mutual information (filter method)

In [ ]:
t0 = time.time()
mi = fs.mutual_information(tr[CANDIDATES], y_tr)
print(f"computed in {time.time()-t0:.0f}s")
mi.to_csv(TAB / "03_mutual_information.csv")
mi.head(15).to_frame()

## 5. Redundancy filter

Each row below is one removed feature, with the near-identical feature that represents it and the reason
(lower MI). This table justifies, feature by feature, why we drop something.

In [ ]:
kept, drops = fs.drop_redundant(tr[CANDIDATES], mi, threshold=0.95)
drops.to_csv(TAB / "03_redundant_features_dropped.csv", index=False)
print(f"{len(CANDIDATES)} candidates -> {len(kept)} non-redundant features ({len(drops)} dropped)")
drops

## 6. Ranking: mutual information + Random Forest importance

In [ ]:
imp = fs.rf_importance(tr[kept], y_tr)
ranking = fs.combined_ranking(mi[kept], imp)
ranking.to_csv(TAB / "03_feature_ranking.csv")

top = ranking.head(25)
fig, axes = plt.subplots(1, 2, figsize=(14, 8), sharey=True)
axes[0].barh(top.index[::-1], top["mutual_information"][::-1], color="#4C72B0"); axes[0].set_title("Mutual information")
axes[1].barh(top.index[::-1], top["rf_importance"][::-1], color="#DD8452"); axes[1].set_title("Random Forest importance")
plt.suptitle("Top 25 features by mean rank (train subsample)")
plt.tight_layout(); plt.savefig(FIG / "03_feature_ranking.png", dpi=150); plt.show()
ranking.head(25)

## 7. How many features? (validation macro-F1)

**Macro-F1** averages the F1 of the 7 families with equal weight, so a k that ignores Web is penalised
as much as one that ignores Benign. We also track **Web F1**, the family the GAN will target.

In [ ]:
ranked = list(ranking.index)
ks = [k for k in (5, 10, 15, 20, 25, 30, 40) if k < len(ranked)] + [len(ranked)]
t0 = time.time()
curve = fs.macro_f1_for_k(ranked, ks, tr, y_tr, va, y_va)
print(f"computed in {time.time()-t0:.0f}s")
K = fs.choose_k(curve)
curve.to_csv(TAB / "03_k_selection_curve.csv", index=False)

plt.figure(figsize=(8, 4.5))
plt.plot(curve["k"], curve["val_macro_f1"], "o-", label="macro-F1 (7 families)")
plt.plot(curve["k"], curve["val_web_f1"], "s--", label="F1 of the Web family")
plt.axvline(K, color="grey", ls=":", label=f"chosen k = {K}")
plt.xlabel("number of features (top-k of the ranking)"); plt.ylabel("validation score")
plt.legend(); plt.grid(alpha=.3); plt.title("Choice of the number of features")
plt.tight_layout(); plt.savefig(FIG / "03_k_selection.png", dpi=150); plt.show()
print(f"Chosen k = {K}")
curve

## 8. The cost (or gain) of the `Dst Port` shortcut

Same model, same k features, with and without `Dst Port`. A large gain from one feature would mean the model leans
on the testbed's port layout. We keep it out of the main models and come back to this in the robustness analysis.

In [ ]:
selected = ranked[:K]
cmp = fs.macro_f1_for_k(selected + ["Dst Port"], [K, K + 1], tr, y_tr, va, y_va)
cmp.insert(0, "features", [f"top-{K} (main)", f"top-{K} + Dst Port"])
cmp.to_csv(TAB / "03_dst_port_comparison.csv", index=False)
cmp

## 9. Save the selection

In [ ]:
path = fs.save_selection(selected, {
    "k": K, "ranking": ranked, "dropped_redundant": drops["dropped"].tolist() if len(drops) else [],
    "excluded": ["Dst Port"], "fitted_on": "train (stratified subsample, <=30k flows per family)"})
print("Saved to", path)
print(f"\n{K} selected features:")
for i, f in enumerate(selected, 1):
    print(f"{i:2d}. {f}")

## 10. What to look at, and what comes next

1. **§3:** where does `Dst Port` rank? And in **§8**, how much macro-F1 does it add? Together they measure the shortcut.
2. **§5:** which features were dropped as redundant? Most pairs are arithmetic identities, e.g. subflow counters equal to
   total counters, because CICFlowMeter rarely splits flows into several subflows.
3. **§6:** do MI and Random Forest agree? Features at the top of *both* lists are the most trustworthy.
4. **§7:** how flat is the curve? A flat curve after k ≈ 15 means most features add nothing.
5. **§9:** the final list, and a **security reading** of each feature. For example, `Init Fwd Win Byts` (initial TCP window)
   reflects the attacking tool's TCP stack, `Fwd Seg Size Min` the packet crafting, and IAT/duration features the
   attack's rhythm (floods are fast, *slow* DoS attacks are deliberately slow).

**Scaling is not done here.** Trees do not need it. For the models that do (Logistic Regression, MLP, GAN), the scaler is
fitted on the training data inside the next steps.

**Next step (04, Vanilla GAN):** train a GAN on the Web training flows (the 597 real ones), check the quality of the
synthetic flows, and build the augmented training sets.

> 📝 Send me the outputs of sections 3, 5, 7, 8 and 9.